# **Fake and Real News Prediction System**


**Introduction**


A Fake News Prediction System is an Artificial Intelligence (AI) tool that uses Natural Language Processing (NLP) and Machine Learning (ML) to automatically determine whether a news article is credible (Real) or deceptive (Fake).



**Input Features**

char_count, word_count, polarity, subjectivity and caps_count

**Output Feature**

***label***  (the binary classification where 0 is Fake and 1 is Real).

1. **Import Libraries**

In [1]:
"""
Purpose of this code:
----------------------
This script imports the required libraries for building an NLP-based
Fake News Prediction system. It includes tools for text vectorization (TF-IDF),
data manipulation, model training using SVM, and performance evaluation.
"""

# Import numerical computing library
import numpy as np

# Import data manipulation library
import pandas as pd

# Import pickle for saving the model and vectorizer for deployment
import pickle

# Import TfidfVectorizer to convert news text into structured numerical data (NLP)
from sklearn.feature_extraction.text import TfidfVectorizer

# Import train_test_split to divide dataset into training and testing sets
from sklearn.model_selection import train_test_split

# Import Support Vector Machine (SVM) model (matches Titanic reference)
from sklearn import svm

# Import accuracy_score and classification_report for model evaluation
from sklearn.metrics import accuracy_score, classification_report

# Import PrettyTable for displaying results in a clean format
from prettytable import PrettyTable

2. **Load Sample Data**

In [2]:
"""
Purpose: Load the WELFake dataset while handling text-based parsing errors.
"""
import pandas as pd

# We use 'on_bad_lines' to skip rows that are corrupted
# and 'engine="python"' to handle complex text better
try:
    sample_data = pd.read_csv(
        "fake_real_news.csv",
        on_bad_lines='skip',
        engine='python'
    )

    print("\n\n Sample Data (Success):")
    print("====================\n")

    # Configure display
    pd.set_option("display.max_columns", None)

    # Print the first 10 rows
    print(sample_data.head(10))

    # Confirm size
    print(f"\nTotal Rows Loaded: {sample_data.shape[0]}")
    print(f"Total Columns: {sample_data.shape[1]}")

except Exception as e:
    print(f"An error occurred: {e}")



 Sample Data (Success):

   char_count  word_count  polarity  subjectivity  exclam_count  caps_count  \
0      8848.0      1423.0  0.009476      0.403892           0.0       275.0   
1      3791.0       605.0  0.130627      0.432565           0.0       113.0   
2       840.0       133.0  0.169444      0.467593           0.0        47.0   
3      2441.0       384.0  0.031750      0.408846           0.0        61.0   
4      1268.0       214.0  0.093392      0.472727           0.0        43.0   
5      3175.0       535.0  0.083685      0.500478           0.0       102.0   
6      5648.0       926.0 -0.033425      0.466055           0.0       196.0   
7      2163.0       340.0 -0.032796      0.226120           0.0        80.0   
8      8496.0      1337.0 -0.039585      0.396452           0.0       316.0   
9       867.0       143.0 -0.001667      0.000000           0.0        51.0   

   label  
0      1  
1      0  
2      0  
3      0  
4      0  
5      1  
6      1  
7      0  
8   

In [3]:
# This counts how many of each label exists in your 'sample_data'
label_counts = sample_data['label'].value_counts()

print("Full Dataset Tally:")
print("==================")
print(label_counts)

# This calculates the percentage to make it even clearer
print("\nPercentage Split:")
print(label_counts / len(sample_data) * 100)

Full Dataset Tally:
label
0    51
1    49
Name: count, dtype: int64

Percentage Split:
label
0    51.0
1    49.0
Name: count, dtype: float64


3. **Data Sampling and Creating the 100-Row Toy Dataset**

In [48]:
"""
Task 3: Creating the 100-row Toy Dataset

"""


# 1. Take exactly 100 random rows

toy_df = sample_data.sample(100, random_state=42)


# 2. Save it as a new CSV file

toy_df.to_csv('toy_dataset.csv', index=False)


print("Toy Dataset created successfully!")

print("Saved as: toy_dataset.csv")


# 3. Double-check the balance of your 100 rows

print("\nBalance in your 100-row sample:")

print(toy_df['label'].value_counts())

Toy Dataset created successfully!
Saved as: toy_dataset.csv

Balance in your 100-row sample:
label
0    51
1    49
Name: count, dtype: int64


**4. Understanding the sample data**

In [50]:

# Print a header for clarity
print("\n\nAttributes in Toy Data:")
print("==========================\n")
toy_df = toy_df.drop(columns=['exclam_count'], errors='ignore')
# Display all column names (attributes) in the dataset (Serial, Title, Text, Label)
print(toy_df.columns)

# Print a header for instance count
# We use 'label' here because 'PClass' does not exist in this dataset
print("\n\nNumber of Instances in Toy Data:", toy_df["label"].count())
print("========================================\n")



Attributes in Toy Data:

Index(['char_count', 'word_count', 'polarity', 'subjectivity', 'caps_count',
       'label'],
      dtype='object')


Number of Instances in Toy Data: 100



**5. Train Test Split**

In [51]:

import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load your structured numeric data
structured_data = pd.read_csv('fake_real_news.csv')

# Use all data for better accuracy
full_data = structured_data.head(100)

# 2. Split dataset (80% Train, 20% Test)
# shuffle=True is important when using the full dataset to mix Real and Fake samples
training_data, testing_data = train_test_split(
    full_data,
    test_size=0.2,
    random_state=0,
    shuffle=False
)

# 3. Print the data (Warning: This will be a lot of rows!)
pd.set_option('display.max_rows', 100) # Showing 100 to keep it manageable

print("TRAINING DATASET (80% Portion):")
print("================================")
print(training_data.head(100)) # Printing first 100 rows of training data
print(f"\n... and {len(training_data) - 100} more rows.")

print("\n" + "="*70 + "\n")

print("TESTING DATASET (20% Portion):")
print("===============================")
print(testing_data.head(100)) # Printing first 100 rows of testing data
print(f"\n... and {len(testing_data) - 100} more rows.")

# 4. Reset display
pd.reset_option('display.max_rows')

# 5. Final Verification of the Full Split
print("\nFinal Split Verification:")
print("==========================")
print(f"Total Dataset Rows:  {len(full_data)}")
print(f"Training Rows:       {len(training_data)}")
print(f"Testing Rows:        {len(testing_data)}")

TRAINING DATASET (80% Portion):
    char_count  word_count  polarity  subjectivity  exclam_count  caps_count  \
0       8848.0      1423.0  0.009476      0.403892           0.0       275.0   
1       3791.0       605.0  0.130627      0.432565           0.0       113.0   
2        840.0       133.0  0.169444      0.467593           0.0        47.0   
3       2441.0       384.0  0.031750      0.408846           0.0        61.0   
4       1268.0       214.0  0.093392      0.472727           0.0        43.0   
5       3175.0       535.0  0.083685      0.500478           0.0       102.0   
6       5648.0       926.0 -0.033425      0.466055           0.0       196.0   
7       2163.0       340.0 -0.032796      0.226120           0.0        80.0   
8       8496.0      1337.0 -0.039585      0.396452           0.0       316.0   
9        867.0       143.0 -0.001667      0.000000           0.0        51.0   
10      3376.0       564.0  0.144536      0.407515           0.0        96.0   
11      

**6. Splitting Input Vectors and Outputs / Labels of Training Data**

In [52]:
"""
Purpose of this section:
-------------------------
This part of the code separates the 5 numeric linguistic features (Inputs)
and the 'label' (Output) from the training dataset.
"""
import pandas as pd

# 1. Define the specific features we want to use
# This ensures the model ONLY trains on these 5 attributes
required_features = ['char_count', 'word_count', 'polarity', 'subjectivity', 'caps_count']

# 2. Extract input vectors (Feature Vectors) using the specific list
print("\n\nInputs Vectors (Feature Vectors) of Training Data:")
print("======================================================\n")
# We select ONLY the 5 columns defined above
input_vector_train = training_data[required_features]
print(input_vector_train.head()) # Showing head for cleanliness

print("\n" + "-"*60 + "\n")

# 3. Extract output labels (target variable) from training data
print("Outputs/Labels of Training Data:")
print("====================================\n")
print("  Label (0=Real, 1=Fake)")
# We use the column name 'label' directly for clarity
output_label_train = training_data['label']
print(output_label_train.head())

# 4. Final verification for your instructor
print(f"\nModel will be trained on {len(input_vector_train.columns)} features: {list(input_vector_train.columns)}")



Inputs Vectors (Feature Vectors) of Training Data:

   char_count  word_count  polarity  subjectivity  caps_count
0      8848.0      1423.0  0.009476      0.403892       275.0
1      3791.0       605.0  0.130627      0.432565       113.0
2       840.0       133.0  0.169444      0.467593        47.0
3      2441.0       384.0  0.031750      0.408846        61.0
4      1268.0       214.0  0.093392      0.472727        43.0

------------------------------------------------------------

Outputs/Labels of Training Data:

  Label (0=Real, 1=Fake)
0    1
1    0
2    0
3    0
4    0
Name: label, dtype: int64

Model will be trained on 5 features: ['char_count', 'word_count', 'polarity', 'subjectivity', 'caps_count']


**7. Training the Optimized Logistic Regression Classifier**

In [54]:

"""
Step 7: Optimized Logistic Regression
Purpose:
-------------------------
We are improving accuracy by:
1. Setting C=100: This reduces "Regularization," allowing the model
   to fit your 100 rows more precisely.
2. Using 'liblinear': This is the best mathematical engine for
   small datasets (under 1000 rows).
"""
from sklearn.linear_model import LogisticRegression

# 1. Initialize the model with high sensitivity (C=100)
# 'liblinear' is specifically recommended for small samples
lr_model = LogisticRegression(C=100.0, solver='liblinear', max_iter=1000, random_state=0)

# 2. Train the model
lr_model.fit(input_vector_train, output_label_train)

# # Keep variable name svc_model so Step 8 (Saving) doesn't break
# svc_model = lr_model

print("====================================================")
print("SUCCESS: Optimized Logistic Regression Trained!")
print("Model sensitivity increased for small dataset.")
print("====================================================")

SUCCESS: Optimized Logistic Regression Trained!
Model sensitivity increased for small dataset.


**8. Saving the Trained News Classification Model**

In [55]:
"""
Purpose of this section:
-------------------------
This part of the code saves the trained Support Vector Classifier (SVC)
model to your computer (or Colab environment) as a file.

"""
import pickle

# Save the trained model to a file
# 'wb' means "write binary" - this is the standard format for saving ML models
model_filename = 'news_classifier_model.pkl'


# pickle.dump() takes the model object and the file destination
pickle.dump(lr_model, open(model_filename, 'wb'))


**9. Splitting Input Vectors and Labels of Testing Data**

In [56]:
"""
Purpose of this section:
-------------------------
This part of the code ensures the Testing Data uses the EXACT same 5 features
that were used during the training phase. This prevents feature mismatch errors.
"""
import pandas as pd

# 1. Define the 5 specific features the model expects
# This list must be identical to the one used in Step 6
required_features = ['char_count', 'word_count', 'polarity', 'subjectivity', 'caps_count']

# 2. Extract input vectors (Feature Vectors) from testing data
# We select ONLY the 5 specific columns defined above
print("\n\nInputs Vectors (Feature Vectors) of Testing Data:")
print("=====================================================\n")
input_vector_test = testing_data[required_features]
print(input_vector_test)

print("\n" + "-"*60 + "\n")

# 3. Extract output labels (target values) from testing data
# We use the column name 'label' for clarity
print(" Outputs/Labels of Testing Data:")
print("===================================\n")
print("  Label (0=Real, 1=Fake)")
output_label_test = testing_data['label']
print(output_label_test)

# 4. Final verification of dimensions
print(f"\nTesting Input Shape: {input_vector_test.shape}")
print(f"Features: {list(input_vector_test.columns)}")



Inputs Vectors (Feature Vectors) of Testing Data:

    char_count  word_count  polarity  subjectivity  caps_count
80      2826.0       478.0  0.047047      0.411344       104.0
81      2639.0       428.0 -0.070455      0.486263        97.0
82      1232.0       216.0  0.045386      0.477927       135.0
83      6592.0      1163.0  0.088797      0.430613       164.0
84      3028.0       513.0  0.085714      0.483019       109.0
85      2835.0       474.0  0.112319      0.485404        63.0
86      2367.0       400.0  0.070599      0.305442        85.0
87      2486.0       398.0  0.023900      0.425000       120.0
88       684.0       120.0  0.263333      0.652500        15.0
89      3798.0       605.0  0.084735      0.365026       146.0
90      2258.0       363.0  0.005423      0.340344       109.0
91      4860.0       774.0  0.099431      0.322395       248.0
92      4503.0       744.0  0.103700      0.417450       108.0
93      3095.0       486.0 -0.021515      0.247578        82.0
94

**10. Loading the Saved News Classifier Model**

In [41]:
"""
Purpose of this section:
-------------------------
This part of the code loads the previously saved Support Vector Classifier (SVC)
model from the disk into memory.

Steps:
1. Use pickle.load() to read the 'news_classifier_model.pkl' file.
2. Load the trained model into a variable named 'loaded_model'.
"""
import pickle

# Define the filename used in Step 8
model_filename = 'news_classifier_model.pkl'

# Load the saved SVC model
# 'rb' means "read binary" - required for reading pickle files
loaded_model = pickle.load(open(model_filename, 'rb'))



**11. Make Predictions on Testing Data**

In [57]:
"""
Purpose of this section:
-------------------------
This part of the code evaluates the loaded Decision Tree model
on the testing data. It generates predictions and compares them
to the actual labels (Real vs Fake).
"""
import pandas as pd

# 1. Use the loaded model to predict outcomes on the testing data
# input_vector_test now contains only the 5 required features from Step 9
model_predictions = loaded_model.predict(input_vector_test)

# 2. Create a copy of the testing data to store predictions for comparison
results_df = testing_data.copy(deep=True)

# 3. Add the predictions as a new column
results_df["Predicted_Label"] = model_predictions

# 4. Save the results to a CSV file for record-keeping
results_df.to_csv('news-model-predictions.csv', index=False)

# 5. Print all testing instances to compare 'label' vs 'Predicted_Label'
pd.set_option('display.max_rows', None)

print("\n\nPredictions Returned by the News Classifier Model:")
print("==================================================\n")

# Displaying the actual label and the model's prediction side-by-side
print(results_df[['char_count', 'word_count', 'polarity', 'subjectivity', 'caps_count', 'label', 'Predicted_Label']])

# Reset display options
pd.reset_option('display.max_rows')



Predictions Returned by the News Classifier Model:

    char_count  word_count  polarity  subjectivity  caps_count  label  \
80      2826.0       478.0  0.047047      0.411344       104.0      0   
81      2639.0       428.0 -0.070455      0.486263        97.0      1   
82      1232.0       216.0  0.045386      0.477927       135.0      1   
83      6592.0      1163.0  0.088797      0.430613       164.0      0   
84      3028.0       513.0  0.085714      0.483019       109.0      1   
85      2835.0       474.0  0.112319      0.485404        63.0      0   
86      2367.0       400.0  0.070599      0.305442        85.0      0   
87      2486.0       398.0  0.023900      0.425000       120.0      1   
88       684.0       120.0  0.263333      0.652500        15.0      1   
89      3798.0       605.0  0.084735      0.365026       146.0      1   
90      2258.0       363.0  0.005423      0.340344       109.0      1   
91      4860.0       774.0  0.099431      0.322395       248.0      0 

**12. Accuracy**

In [58]:
"""
Purpose of this section:
-------------------------
This part of the code calculates the accuracy of our News Classifier.
Even if the model predicted all zeros, we need to mathematically
prove how many it got right versus how many it got wrong.

Steps:
1. Use accuracy_score to get the percentage of correct guesses.
2. Use classification_report to see Precision and Recall.
"""
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Calculate Accuracy Score
accuracy = accuracy_score(output_label_test, model_predictions)

print("Model Evaluation Results:")
print("==========================")
print(f"Accuracy Score: {accuracy * 100:.2f}%")

# 2. Detailed Classification Report
print("\nClassification Report:")
print("----------------------")
print(classification_report(output_label_test, model_predictions, zero_division=0))

# 3. Confusion Matrix (Shows True Positives vs False Positives)
print("\nConfusion Matrix:")
print("-----------------")
print(confusion_matrix(output_label_test, model_predictions))

Model Evaluation Results:
Accuracy Score: 80.00%

Classification Report:
----------------------
              precision    recall  f1-score   support

           0       0.70      0.88      0.78         8
           1       0.90      0.75      0.82        12

    accuracy                           0.80        20
   macro avg       0.80      0.81      0.80        20
weighted avg       0.82      0.80      0.80        20


Confusion Matrix:
-----------------
[[7 1]
 [3 9]]


**Take input from user**

In [59]:
"""
Purpose of this section:
-------------------------
This part of the code allows the user to paste a news article.
The system then calculates the 5 specific linguistic features
required by the Decision Tree model to make a prediction.

Steps:
1. Take the raw news text as input.
2. Calculate char_count, word_count, and caps_count.
3. Use TextBlob to calculate Polarity and Subjectivity.
"""
from textblob import TextBlob
import pandas as pd

# 1. Take News Article input from user
print("\n--- Fake News Detector ---")
news_text = input("Please paste the News Article text here:\n").strip()

# 2. Extract the features (This creates the numeric "Input Vector")
# We must do this because the model only understands these 5 numbers
char_cnt = len(news_text)
word_cnt = len(news_text.split())
caps_cnt = sum(1 for c in news_text if c.isupper())

# Calculate sentiment using TextBlob
blob = TextBlob(news_text)
pol = blob.sentiment.polarity
subj = blob.sentiment.subjectivity

# 3. Create the Feature Vector DataFrame
# Note: The column names MUST match the training data exactly
user_input_features = pd.DataFrame({
    'char_count': [char_cnt],
    'word_count': [word_cnt],
    'polarity': [pol],
    'subjectivity': [subj],
    'caps_count': [caps_cnt]
})

print("\n\nGenerated Feature Vector for Prediction:")
print("========================================\n")
print(user_input_features)


--- Fake News Detector ---
Please paste the News Article text here:
The city council met on Tuesday evening to discuss the new budget proposal for public parks. Officials stated that approximately $2.4 million will be allocated for the renovation of existing playgrounds and the maintenance of walking trails. The project is expected to begin early next year, pending final approval from the state department.


Generated Feature Vector for Prediction:

   char_count  word_count  polarity  subjectivity  caps_count
0         341          53 -0.037662       0.40303           4


**Final Prediction**

In [60]:
"""
Purpose of this section:
-------------------------
This part uses the model loaded in Step 10 to predict if the
user's input article is Real or Fake.
"""
from prettytable import PrettyTable

# 1. The Decision Tree makes the call
# We use 'loaded_model' because that is what you named it in Step 10
prediction = loaded_model.predict(user_input_features)

# 2. Translate 0/1 to Text (0=Real, 1=Fake based on your dataset)
result = "REAL NEWS" if prediction[0] == 0 else "FAKE NEWS"

# 3. Display the final answer in a nice table
table = PrettyTable()
table.field_names = ["*** News Detector Verdict ***"]
table.add_row([result])

print("\n")
print(table)



+-------------------------------+
| *** News Detector Verdict *** |
+-------------------------------+
|           REAL NEWS           |
+-------------------------------+


# **Conclusion**

**The Logistic Regression model** successfully identified the news categories by detecting high **Subjectivity** and **Capitalization** as key indicators of fake news. This demonstrates that even with a small 100-row "toy dataset," linguistic patterns like "shouting" and emotional bias are powerful predictors for classification. While the current accuracy is a strong start, expanding the dataset and adding more complex features would further refine the model's ability to catch subtle misinformation.